In [1]:
import pyspark
from pyspark.sql import SparkSession

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA, Imputer
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.sql.functions import mean, col, expr
import numpy as np
import time

In [2]:
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.maxResultSize", "3g") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "25g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "16") \
    .config("spark.shuffle.partitions", "180") \
    .config("spark.kryoserializer.buffer.max", "256m") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.sql.execution.arrow.enabled", "true") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/07/05 16:56:29 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/07/05 16:56:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/05 16:56:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#spark.sparkContext.stop()

In [4]:
parquet_files = ["Parquet/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet", 
                 "Parquet/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet", 
                 "Parquet/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet", 
                 "Parquet/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "Parquet/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet", 
                 "Parquet/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "Parquet/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet", 
                 "Parquet/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet"]

In [5]:
#Read the parquet files
df = spark.read.parquet(*parquet_files, inferSchema=True)

24/07/05 16:56:30 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:56:30 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:56:30 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


In [6]:
#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [7]:
start_time = time.time()

#Drop labels and get remaining counts
labels_to_drop = ['Defense Evasion', 
                  'Exfiltration',
                  'Initial Access',
                  'Lateral Movement', 
                  'Persistence',
                  'Privilege Escalation', 
                  'Resource Development', 
                  'Credential Access']
                  #'Discovery',
                  #'Reconnaissance']

df = df.filter(~col("label_tactic").isin(labels_to_drop))

#Get unique label counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

df_none = df.filter((col("label_tactic") == "none") | (col("label_tactic") == "Reconnaissance"))
label_counts = df_none.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

# Reducing 'label_tactic' - "none" by 95%
df_none = df_none.sample(False, 0.05, seed=42)
label_counts = df_none.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

label_counts = df.groupBy("label_tactic").count().collect()
total_count = df.count()
print("Original Total Count:", total_count)

# Calculate sampling fractions/proportions for each stratum/category (Doing Stratified Sampling
fractions = {row['label_tactic']: row['count'] / total_count for row in label_counts}

print(fractions)


df = df.filter(~((col("label_tactic") == "none") | (col("label_tactic") == "Reconnaissance")))
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

df = df_none.union(df)
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")
label_counts.show()

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|     Discovery|   2086|
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+



+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+



+--------------+------+
|  label_tactic| count|
+--------------+------+
|Reconnaissance|463155|
|          none|463354|
+--------------+------+



Original Total Count: 18562407
{'none': 0.5000213065040542, 'Discovery': 0.00011237766740056934, 'Reconnaissance': 0.4998663158285453}
+------------+-----+
|label_tactic|count|
+------------+-----+
|   Discovery| 2086|
+------------+-----+



+--------------+------+
|  label_tactic| count|
+--------------+------+
|     Discovery|  2086|
|Reconnaissance|463155|
|          none|463354|
+--------------+------+

Execution time: 8.479144096374512 seconds


In [8]:
#Drop uid feature
df = df.drop('uid')

In [9]:
start_time = time.time()

#Columns to index
columns_to_index = ['service', 
                    'conn_state', 
                    'history', 
                    'proto', 
                    'dest_ip_zeek', 
                    'community_id', 
                    'src_ip_zeek',
                    'datetime',
                    'local_resp',
                    'local_orig',
                    'label_tactic']

#Cast datetime, local_resp, local_orig to String
df = df.withColumn("datetime", col("datetime").cast("string"))
df = df.withColumn("local_resp", col("local_resp").cast("string"))
df = df.withColumn("local_orig", col("local_orig").cast("string"))

#Impute null values with empty string
for column in columns_to_index:
    df = df.fillna('', subset=[column])

In [10]:
#Split the into training and test sets
start_time = time.time()
train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.02176499366760254 seconds


In [11]:
#StringIndexer
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").setHandleInvalid("keep") for column in columns_to_index]

#Chain indexers together
pipeline = Pipeline(stages=indexers).fit(train_data)

#Fit and transform the data
train_data_indexed = pipeline.transform(train_data)
test_data_indexed = pipeline.transform(test_data)

#Drop original columns
train_data_indexed = train_data_indexed.drop(*columns_to_index)
train_data_indexed = train_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")
test_data_indexed = test_data_indexed.drop(*columns_to_index)
test_data_indexed = test_data_indexed.withColumnRenamed("label_tactic_indexed", "label_tactic")

#print("train_indexed columns: ", train_data_indexed.columns)
#print("test_indexed columns: ", test_data_indexed.columns)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 40.01737833023071 seconds


In [12]:
start_time = time.time()

#List of numeric column names
numeric_columns = ['resp_pkts', 
                   'orig_ip_bytes', 
                   'missed_bytes', 
                   'duration', 
                   'orig_pkts',
                   'resp_ip_bytes', 
                   'dest_port_zeek', 
                   'orig_bytes', 
                   'resp_bytes',
                   'src_port_zeek', 
                   'ts']

#Create Imputer
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

#Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data_indexed)

#Apply the Imputer to the training data
train_data_imputed = imputer_model.transform(train_data_indexed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Apply the Imputer to the test data
start_time = time.time()
test_data_imputed = imputer_model.transform(test_data_indexed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

train_data_imputed = train_data_imputed.drop(*numeric_columns)
test_data_imputed = test_data_imputed.drop(*numeric_columns)

#print("\nTrain data imputed: ", train_data_imputed.columns)
#print("\n")
#print("Test data imputed: ", test_data_imputed.columns)

Execution time: 3.2610528469085693 seconds
Execution time: 0.01839590072631836 seconds


In [13]:
start_time = time.time()

#Create VectorAssembler
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed") or column.endswith("_indexed")]
#print("Columns to assemble: ", columns_to_assemble)

assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

#Transform the training data
train_data_assembled = assembler.transform(train_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Transform the test data
start_time = time.time()
test_data_assembled = assembler.transform(test_data_imputed)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Select the features and label columns
train_data_assembled = train_data_assembled.select("features", "label_tactic")
test_data_assembled = test_data_assembled.select("features", "label_tactic")

#print("Train_data_assembled columns: ", train_data_assembled.columns)
#print("Test_data_assembled columns: ", test_data_assembled.columns)

Execution time: 0.6781466007232666 seconds
Execution time: 0.2613799571990967 seconds


In [14]:
from pyspark.ml.feature import StandardScaler

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic")

24/07/05 16:57:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
24/07/05 16:57:31 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 16:57:39 WARN DAGScheduler: Broadcasting large task binary with size 21.4 MiB


In [15]:
# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic")

In [16]:
# Define the PCA model
pca = PCA(k=5, inputCol="features_normalized", outputCol="pca_features")

# Fit the PCA model on the normalized training set
start_time = time.time()
pca_model = pca.fit(train_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 16:57:41 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 16:57:42 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:57:44 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 16:57:44 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:57:46 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 16:57:46 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:57:46 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been depre

Execution time: 21.616084098815918 seconds


24/07/05 16:58:01 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [17]:
# Apply PCA transformation to the training and test sets
train_pca = pca_model.transform(train_data_normalized)
test_pca = pca_model.transform(test_data_normalized)

In [18]:
# Drop the normalized column and rename the pca_features column
train_pca = train_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")
test_pca = test_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")

# Verify the changes
#train_pca.show()
#test_pca.show()

In [19]:
#Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)
ovr = OneVsRest(classifier=svm, labelCol="label_tactic")
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.013194799423217773 seconds


In [20]:
#Fit the model
start_time = time.time()
svm_model = ovr.fit(train_pca)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 16:58:05 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:58:05 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 16:58:12 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 16:58:12 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:58:12 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 16:58:12 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.py

Execution time: 270.3898813724518 seconds


In [21]:
#Make predictions
start_time = time.time()

predictions = svm_model.transform(test_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 17:02:32 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.


Execution time: 0.4631040096282959 seconds


In [22]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

#Calculate FPR
evaluator_fprL = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="falsePositiveRateByLabel")
fprL_score = evaluator_fprL.evaluate(predictions)

#Calculate Weighted FPR
evaluator_fpr = MulticlassClassificationEvaluator(labelCol="label_tactic", metricName="weightedFalsePositiveRate")
fpr_score = evaluator_fpr.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)
print("FPR by Label:", fprL_score)
print("Weighted FPR:", fpr_score)

24/07/05 17:02:34 WARN DAGScheduler: Broadcasting large task binary with size 21.6 MiB
24/07/05 17:02:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 17:02:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 17:02:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 17:02:34 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 17:02:34 WARN SQLConf: The SQL config 'spark.sql

Accuracy: 0.9793827391042031
Precision: 0.9773836134418896
Recall: 0.9793827391042031
F1-Score: 0.9782911995349883
FPR by Label: 0.029087261785356068
Weighted FPR: 0.02052113036930317


In [23]:
start_time = time.time()

#Extract predictions and labels
predictions_and_labels = predictions.select("prediction", "label_tactic")

#Calculate false positives and true negatives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic== 0)).count()
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic == 0)).count()

#Calculate FPR
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/07/05 17:03:41 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB
24/07/05 17:03:50 WARN DAGScheduler: Broadcasting large task binary with size 21.5 MiB


False Positive Rate: 0.007697514329329762
Execution time: 18.139090538024902 seconds


In [24]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import Row

# Convert DataFrame to RDD of tuples (prediction, label)
prediction_and_labels = predictions.select("prediction", "label_tactic") \
    .rdd.map(lambda row: (float(row['prediction']), float(row['label_tactic'])))


# Instantiate BinaryClassificationMetrics
metrics = BinaryClassificationMetrics(prediction_and_labels)

# Compute AUROC
auROC = metrics.areaUnderROC

# Print AUROC
print("Area under ROC = ", auROC)

/home/colin/.local/lib/python3.10/site-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
24/07/05 17:03:59 WARN DAGScheduler: Broadcasting large task binary with size 21.6 MiB
24/07/05 17:03:59 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 17:04:01 WARN DAGScheduler: Broadcasting large task binary with size 21.6 MiB
24/07/05 17:04:02 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/05 17:04:02 WARN SQLConf: The SQL config 'spark.sql.execution.arrow.enabled' has been deprecated in Spark v3.0 and may be removed in the future. Use 'spark.sql.execution.arrow.pyspark.enabled' instead of it.
24/07/0

Area under ROC =  0.9816076119426571


In [25]:
spark.sparkContext.stop()